# CIFAR-10 MobileNetV2 baseline (Kaggle)

This notebook retrains the FP32 baseline on a deterministic 45,000/5,000 split of CIFAR-10's official training set. The validation split selects the checkpoint; the official test set is touched only in the final evaluation cell.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret('GITHUB_TOKEN')
username, repo_name = 'AdiGiriIIT', 'CS6886--Assignment-2'
!git clone https://{token}@github.com/{username}/{repo_name}.git assignment-2
%cd assignment-2
!python -m pip install -q PyYAML matplotlib

In [ ]:
from pathlib import Path

DATA_DIR = '/kaggle/input/datasets/adityagirishep23b048/cifar-data'
cifar_dir = Path(DATA_DIR) / 'cifar-10-batches-py'
required = ['data_batch_1', 'data_batch_2', 'data_batch_3', 'data_batch_4', 'data_batch_5', 'test_batch', 'batches.meta']
assert all((cifar_dir / name).is_file() for name in required), f'Missing CIFAR-10 files under {cifar_dir}'
print(f'Using CIFAR-10 at {cifar_dir}')
print('Split: 45,000 train / 5,000 validation (seed 6886); official test set remains held out.')

In [ ]:
# Smoke test: validation is used here; the held-out test set is not evaluated.
!set -o pipefail; python -m src.train --config configs/baseline.yaml --data-dir "{DATA_DIR}" --device cuda --epochs 1 --max-train-batches 5 --max-val-batches 2 --run-name sanity 2>&1 | tee results/logs/baseline-sanity.log

In [ ]:
# Full run. baseline.pt is the checkpoint with the best validation accuracy.
!set -o pipefail; python -m src.train --config configs/baseline.yaml --data-dir "{DATA_DIR}" --device cuda 2>&1 | tee results/logs/baseline-train.log

In [ ]:
from IPython.display import Image, display

curve = Path('results/curves/baseline.png')
assert curve.is_file(), f'Missing training plot: {curve}'
display(Image(filename=str(curve)))

In [ ]:
# The only baseline test-set evaluation. Do this after validation-based model selection.
!set -o pipefail; python -m src.evaluate --checkpoint results/checkpoints/baseline.pt --data-dir "{DATA_DIR}" --device cuda 2>&1 | tee results/logs/baseline-held-out-test.log

In [ ]:
import hashlib
import tarfile
from IPython.display import FileLink

checkpoint = Path('results/checkpoints/baseline.pt')
assert checkpoint.is_file(), 'Baseline checkpoint was not created.'
print('baseline.pt SHA-256:', hashlib.sha256(checkpoint.read_bytes()).hexdigest())
archive = Path('baseline-artifacts.tgz')
with tarfile.open(archive, 'w:gz') as tar:
    tar.add(checkpoint, arcname=str(checkpoint))
    tar.add('results/curves/baseline.csv', arcname='results/curves/baseline.csv')
    tar.add('results/curves/baseline.png', arcname='results/curves/baseline.png')
    tar.add('results/logs/baseline-held-out-test.log', arcname='results/logs/baseline-held-out-test.log')
print(f'Created {archive} ({archive.stat().st_size:,} bytes)')
FileLink(str(archive))